# AWS AASTU Cohort - Week 02: Mini Semantic Search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eyustarX/data-processing-app/blob/main/week-02/task.ipynb)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)
[![ChromaDB](https://img.shields.io/badge/ChromaDB-Vector_Store-red.svg)](https://www.trychroma.com/)

---

## 🎯 Lab Objectives
In this hands-on lab, we build a **Mini Semantic Search Engine** over an AWS cloud and AI/ML FAQ knowledge base:
1. **Load a CSV/JSON of FAQs with Pandas**
2. **Clean it and chunk the text** for optimal retrieval
3. **Embed and store it in Chroma** vector database
4. **Query it and print the top 3 matches** with similarity rankings


### Step 1: Install & Import Dependencies
Ensure `chromadb`, `pandas`, and `numpy` are installed.

In [1]:
# Install dependencies if executing in Google Colab
# !pip install -q chromadb pandas numpy sentence-transformers

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Import modular week 2 semantic search functions
from semantic_search import (
    load_faq_data,
    clean_and_chunk_faqs,
    init_vector_store,
    semantic_search,
    format_search_results,
)
print("Libraries loaded successfully. Pandas, NumPy, and Vector Engine ready.")

Libraries loaded successfully. Pandas, NumPy, and Vector Engine ready.


### Step 2: Load FAQ Dataset with Pandas
We ingest the curated AWS & AI/ML FAQ dataset (`data/faqs.json` or `data/faqs.csv`).

In [2]:
df = load_faq_data("data/faqs.json")
print(f"Loaded {len(df)} FAQ records.")
print(f"Columns: {list(df.columns)}")
df.head()

Loaded 12 FAQ records.
Columns: ['id', 'category', 'question', 'answer']


        id             category                                           question                                             answer
0  faq-001          AWS Compute          What is Amazon EC2 and when should I use it?  Amazon Elastic Compute Cloud (Amazon EC2) prov...
1  faq-002          AWS Storage  How does Amazon S3 storage classes and pricing work?  Amazon Simple Storage Service (Amazon S3) is a...
2  faq-003       AWS Serverless  What is AWS Lambda and how does serverless execu...   AWS Lambda is a serverless, event-driven compu...
3  faq-004         AWS Security  How do I secure my AWS infrastructure using IAM ...   AWS Identity and Access Management (IAM) contr...
4  faq-005            AWS AI/ML  What is Amazon SageMaker and what are its key ca...   Amazon SageMaker is a fully managed machine le...

### Step 3: Text Cleaning & Chunking
We clean the text by stripping extraneous whitespace, normalizing casing, and splitting documents into contextual overlapping passages.

In [3]:
chunks = clean_and_chunk_faqs(df, max_words=60)
print(f"Generated {len(chunks)} total chunks.")
sample = chunks[0]
print("Chunk Example:")
print(f"  ID: {sample['id']}")
print(f"  Category: {sample['category']}")
print(f"  Text Preview: {sample['text'][:350]}")

Generated 15 total chunks.
Chunk Example:
  ID: faq-001_chunk_0
  Category: AWS Compute
  Text Preview: Question: What is Amazon EC2 and when should I use it? Answer: Amazon Elastic Compute Cloud (Amazon EC2) provides scalable virtual servers (instances) in the cloud. You should use EC2 when you need complete control over your operating system, custom software stacks, high-performance compute clusters, or continuous background processing.


### Step 4: Embed & Store in Chroma Vector Store
We initialize the vector store, compute dense embeddings for all chunks, and index them alongside rich metadata.

In [4]:
vector_store = init_vector_store(chunks, collection_name="aws_faqs")
print(f"Vector store collection initialized and populated with {len(chunks)} chunk embeddings.")

Vector store collection initialized and populated with 15 chunk embeddings.


### Step 5: Query & Print Top 3 Matches
We execute semantic similarity queries across diverse user questions, printing the top 3 ranked results.

In [5]:
queries = [
    "How is pricing calculated for S3 bucket storage?",
    "What tools exist for deploying machine learning models on AWS?",
    "How do I apply the principle of least privilege in cloud security?",
]

for q in queries:
    results = semantic_search(vector_store, q, top_k=3)
    print(format_search_results(q, results))



[Query] 'How is pricing calculated for S3 bucket storage?'
----------------------------------------------------------------------
[Rank #1] | Category: AWS Storage | Similarity: 48.59% (Distance: 0.5141)
  * Question: How does Amazon S3 storage classes and pricing work?
  * Answer  : Amazon Simple Storage Service (Amazon S3) is an object storage service offering 99.999999999% (11 9s) data durability. Pricing is based on storage volume (GB/month), requests (GET, PUT), and data transfer out. Classes include S3 Standard for frequent access, S3 Intelligent-Tiering for automatic cost optimization, and S3 Glacier for long-term archiving.

[Rank #2] | Category: AWS Storage | Similarity: 25.14% (Distance: 0.7486)
  * Question: How does Amazon S3 storage classes and pricing work?
  * Answer  : Amazon Simple Storage Service (Amazon S3) is an object storage service offering 99.999999999% (11 9s) data durability. Pricing is based on storage volume (GB/month), requests (GET, PUT), and data transfe

### Step 6: Interactive Search Demo
Try your own query!

In [6]:
user_query = "What is Chroma and why is chunking used?"
top_matches = semantic_search(vector_store, user_query, top_k=3)
print(format_search_results(user_query, top_matches))



[Query] 'What is Chroma and why is chunking used?'
----------------------------------------------------------------------
[Rank #1] | Category: Vector DB & RAG | Similarity: 33.72% (Distance: 0.6628)
  * Question: What is ChromaDB and how does it store vector embeddings?
  * Answer  : Chroma is an open-source, AI-native embedding database (vector store) designed to make building LLM apps simple. Chroma stores high-dimensional vector representations of text documents alongside metadata and raw document text, enabling sub-millisecond approximate nearest neighbor (ANN) similarity search.

[Rank #2] | Category: Vector DB & RAG | Similarity: 28.51% (Distance: 0.7149)
  * Question: Why is text chunking important in Retrieval Augmented Generation (RAG)?
  * Answer  : Text chunking breaks large documents into smaller, coherent passages before generating embeddings. Proper chunking prevents dilution of semantic meaning across large texts, fits within embedding model context limits, and ensures